# 09 Facade | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: zlozone podsystemy
2. 🏛️ Fasada jako punkt wejscia
3. 🔗 Implementacja: delegacja do podsystemow
4. 🔀 Wiele fasad
5. 🆚 Facade vs Adapter vs Mediator

## 1. 🔹 Problem: zlozone podsystemy

Fasada (Facade) to wzorzec strukturalny upraszczajacy dostep do
zlozonego podsystemu poprzez udostepnienie prostego interfejsu.

Analogica: recepcja hotelowa - klient nie musi wiedziec jak
dziala kuchnia, sprzataczki, ksiegowosc. Mowi tylko do recepcji.

Problem bez Fasady:
- Klient musi znac wiele klas i ich kolejnosc wywolywania
- Zmiana podsystemu wymaga zmian wszytstkich klientow
- Wysoki coupling miedzy klientem a podsystemem

Fasada rozwiazuje:
- Jeden punkt wejscia do podsystemu
- Klient nie musi znac szczegolów implementacji
- Zmiany w podsystemie nie dotykaja klienta

> 💡 Fasada upraszcza INTERFEJS. Nie zabrania dostep do
> podsystemow bezposrednio - to nie proxy i nie enkapsulacja.

In [ ]:
# Skomplikowany podsystem odtwarzacza wideo
class VideoDecoder:
    def decode(self, filename: str) -> str:
        print(f'Decoding video: {filename}')
        return f'decoded:{filename}'

class AudioDecoder:
    def decode(self, filename: str) -> str:
        print(f'Decoding audio: {filename}')
        return f'audio:{filename}'

class SubtitleLoader:
    def load(self, filename: str) -> str:
        print(f'Loading subtitles: {filename}')
        return f'subs:{filename}'

class VideoRenderer:
    def render(self, video_data: str) -> None:
        print(f'Rendering: {video_data}')

class AudioRenderer:
    def play(self, audio_data: str) -> None:
        print(f'Playing: {audio_data}')

# BEZ fasady: klient musi zarzadzac wszystkimi podsystemami
print('--- Bez fasady ---')
video_dec = VideoDecoder()
audio_dec = AudioDecoder()
video_ren = VideoRenderer()
audio_ren = AudioRenderer()

filename = 'movie.mp4'
video = video_dec.decode(filename)
audio = audio_dec.decode(filename)
video_ren.render(video)
audio_ren.play(audio)
print('Klient musi znac 4 klasy i ich kolejnosc!')

---

### 🐍 Cwiczenia - problem

1. Policz ile klas musi znac klient w przykladzie bez fasady.
   Ile linii kodu wykonuje uzytkownik vs z fasada?
2. Napisz liste 3 przykladow z zycia codziennego gdzie fasada
   upraszcza dostep do zlozonego systemu.
3. *(Trudniejsze)* Zidentyfikuj fasady w bibliotece standardowej
   Pythona - `smtplib`, `zipfile`, `sqlite3`. Opisz jakie
   podsystemy ukrywaja.

In [ ]:
# Cwiczenie 1: analiza zlozonosci
klasy_bez_fasady = ['VideoDecoder', 'AudioDecoder', 'VideoRenderer', 'AudioRenderer']
linii_kodu_klienta = 8  # w przykladzie powyzej
linii_z_fasada = 1      # player.play('movie.mp4')

print(f'Klas do poznania: {len(klasy_bez_fasady)}')
print(f'Linii kodu klienta: {linii_kodu_klienta}')
print(f'Z fasada: {linii_z_fasada} linia')
print(f'Redukcja: {linii_kodu_klienta / linii_z_fasada}x')

In [ ]:
# Cwiczenie 2: przyklady z zycia
examples = [
    'Recepcja hotelowa: ukrywa kuchnie, sprzataczki, ksiegowosc',
    'Pilot TV: ukrywa tunery, dekodery, systemy audio',
    'Interfejs aplikacji mobilnej: ukrywa serwery, bazy danych, mikroserwisy',
]
for i, ex in enumerate(examples, 1):
    print(f'{i}. {ex}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: fasady w stdlib
facades = {
    'smtplib.SMTP': 'Ukrywa protokol SMTP (EHLO, MAIL FROM, RCPT TO, DATA)',
    'zipfile.ZipFile': 'Ukrywa kompresje deflate/bzip2, naglowki ZIP',
    'sqlite3': 'Ukrywa silnik SQLite: parser SQL, B-tree, WAL',
}
for facade, description in facades.items():
    print(f'{facade}:\n  {description}\n')

## 2. 🔹 Fasada jako punkt wejscia

Struktura Fasady:
- Facade: glowna klasa udostepniajaca uproszczone API
- Subsystem classes: klasy implementujace logike biznesowa

Fasada:
- Tworzy lub otrzymuje instancje podsystemow
- Deleguje prace do podsystemow
- Nie implementuje logiki biznesowej samodzielnie
- Moze przyjmowac podsystemy jako zaleznosci (dependency injection)

Klient:
- Komunikuje sie tylko z fasada
- Opcjonalnie moze korzystac z podsystemow bezposrednio

> 💡 Fasada jest prostszym wzorcem niz Adapter czy Proxy.
> Nie zmienia interfejsu podsystemu - tworzy nowy, prostszy interfejs.

In [ ]:
# Fasada odtwarzacza wideo - prosty interfejs
class VideoPlayerFacade:
    def __init__(self) -> None:
        self._video_dec = VideoDecoder()
        self._audio_dec = AudioDecoder()
        self._subtitle_loader = SubtitleLoader()
        self._video_ren = VideoRenderer()
        self._audio_ren = AudioRenderer()

    def play(self, filename: str, subtitles: bool = False) -> None:
        print(f'--- Playing {filename} ---')
        video = self._video_dec.decode(filename)
        audio = self._audio_dec.decode(filename)
        if subtitles:
            self._subtitle_loader.load(filename + '.srt')
        self._video_ren.render(video)
        self._audio_ren.play(audio)
        print('--- Done ---')

    def stop(self) -> None:
        print('Stopped')

# Klient - prosty interfejs
player = VideoPlayerFacade()
player.play('movie.mp4', subtitles=True)
player.stop()

---

### 🐍 Cwiczenia - implementacja fasady

1. Napisz `EmailFacade.send(to, subject, body)` ukrywajaca
   `SMTPConnection`, `EmailFormatter`, `SpamFilter`.
2. Napisz `ReportFacade.generate(data, format)` ukrywajaca
   `DataProcessor`, `ChartBuilder`, `PDFExporter`.
3. *(Trudniejsze)* Napisz fasade z dependency injection:
   `SystemFacade(__init__(self, db, cache, logger))` przyjmujaca
   zaleznosci zamiast tworzac je samodzielnie.

In [ ]:
# Cwiczenie 1: EmailFacade
class SMTPConnection:
    def connect(self, host: str) -> None: print(f'Connected to {host}')
    def send(self, from_: str, to: str, msg: str) -> None: print(f'Sent to {to}')
    def disconnect(self) -> None: print('Disconnected')

class EmailFormatter:
    def format(self, subject: str, body: str) -> str:
        return f'Subject: {subject}\n\n{body}'

class SpamFilter:
    def check(self, content: str) -> bool: return 'spam' not in content.lower()

class EmailFacade:
    def send(self, to: str, subject: str, body: str) -> bool:
        ...

facade = EmailFacade()
facade.send('alice@example.com', 'Hello', 'How are you?')

In [ ]:
# Cwiczenie 2: ReportFacade
class DataProcessor:
    def process(self, data: list) -> dict:
        return {'rows': len(data), 'sum': sum(data)}

class ChartBuilder:
    def build(self, stats: dict) -> str:
        return f'Chart({stats})'

class PDFExporter:
    def export(self, content: str, filename: str) -> None:
        print(f'Exported to {filename}')

class ReportFacade:
    def generate(self, data: list, filename: str) -> None:
        ...

ReportFacade().generate([10, 20, 30, 40], 'report.pdf')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: fasada z DI
class Database:
    def query(self, sql: str) -> list: return [{'id': 1}]

class Cache:
    def __init__(self): self._store = {}
    def get(self, key: str): return self._store.get(key)
    def set(self, key: str, value): self._store[key] = value

class AppLogger:
    def info(self, msg: str): print(f'[INFO] {msg}')

class SystemFacade:
    def __init__(self, db: Database, cache: Cache, logger: AppLogger):
        ...
    def get_users(self) -> list:
        ...

facade = SystemFacade(Database(), Cache(), AppLogger())
print(facade.get_users())
print(facade.get_users())  # z cache

## 3. 🔹 Implementacja: delegacja do podsystemow

Fasada implementuje metody delegujace prace do podsystemow.
Kluczowe wzorce implementacyjne:

1. Orkiestracja - fasada zarzadza kolejnoscia wywolan
2. Agregacja wynikow - fasada zbiera wyniki z wielu podsystemow
3. Obsługa bledow - fasada centralizuje try/except
4. Logowanie - jedno miejsce do dodania logowania

Fasada jako context manager (`with` statement):
- `__enter__` - inicjalizacja podsystemow
- `__exit__` - sprzatanie (zamkniecie polaczen)

Fasada moze byc Singletonem:
- Jeden punkt wejscia dla calego podsystemu
- Dzielona przez wiele klientow

In [ ]:
# Fasada bazy danych z obsługa bledow i context manager
class ConnectionPool:
    def get_connection(self) -> str:
        print('Getting connection from pool')
        return 'conn:1'
    def release(self, conn: str) -> None:
        print(f'Releasing {conn}')

class QueryExecutor:
    def execute(self, conn: str, sql: str, params: tuple = ()) -> list:
        print(f'SQL: {sql} params={params}')
        return [{'id': 1, 'name': 'Alice'}, {'id': 2, 'name': 'Bob'}]

class TransactionManager:
    def begin(self, conn: str) -> None: print('BEGIN TRANSACTION')
    def commit(self, conn: str) -> None: print('COMMIT')
    def rollback(self, conn: str) -> None: print('ROLLBACK')

class DatabaseFacade:
    def __init__(self) -> None:
        self._pool = ConnectionPool()
        self._executor = QueryExecutor()
        self._tx = TransactionManager()

    def query(self, sql: str, params: tuple = ()) -> list:
        conn = self._pool.get_connection()
        try:
            return self._executor.execute(conn, sql, params)
        finally:
            self._pool.release(conn)

    def execute_transaction(self, queries: list) -> bool:
        conn = self._pool.get_connection()
        self._tx.begin(conn)
        try:
            for sql, params in queries:
                self._executor.execute(conn, sql, params)
            self._tx.commit(conn)
            return True
        except Exception as e:
            print(f'Error: {e}')
            self._tx.rollback(conn)
            return False
        finally:
            self._pool.release(conn)

db = DatabaseFacade()
users = db.query('SELECT * FROM users')
print(f'Users: {users}')

success = db.execute_transaction([
    ('INSERT INTO log VALUES (?)', ('login',)),
    ('UPDATE users SET active=1 WHERE id=?', (1,)),
])
print(f'Transaction: {"OK" if success else "FAILED"}')

---

### 🐍 Cwiczenia - delegacja

1. Rozszerz `DatabaseFacade` o metode `bulk_insert(table, rows)`
   ktora wykona transakcje z wieloma INSERT.
2. Napisz `FileSystemFacade` z metodami `read(path)`, `write(path, content)`,
   `copy(src, dst)` ukrywajaca operacje `open/read/write/os`.
3. *(Trudniejsze)* Napisz `CachedDatabaseFacade(db, cache)` ktora
   dla zapytan SELECT uzywa cache z TTL=60s.

In [ ]:
# Cwiczenie 1: bulk_insert
class DatabaseFacadeExt(DatabaseFacade):
    def bulk_insert(self, table: str, rows: list[dict]) -> int:
        ...

db_ext = DatabaseFacadeExt()
inserted = db_ext.bulk_insert('products', [
    {'name': 'Widget', 'price': 9.99},
    {'name': 'Gadget', 'price': 19.99},
])
print(f'Inserted: {inserted} rows')

In [ ]:
# Cwiczenie 2: FileSystemFacade
import os

class FileSystemFacade:
    def read(self, path: str) -> str:
        ...
    def write(self, path: str, content: str) -> None:
        ...
    def copy(self, src: str, dst: str) -> None:
        ...

fs = FileSystemFacade()
fs.write('/tmp/test_facade.txt', 'Hello, Facade!')
content = fs.read('/tmp/test_facade.txt')
print(f'Read: {content}')
fs.copy('/tmp/test_facade.txt', '/tmp/test_facade_copy.txt')
print('Copied!')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: CachedDatabaseFacade
import time

class CachedDatabaseFacade:
    TTL = 60

    def __init__(self, db: DatabaseFacade, cache: dict = None):
        self._db = db
        self._cache: dict[str, tuple] = cache or {}

    def query(self, sql: str, params: tuple = ()) -> list:
        # hint: klucz = (sql, params), wartosc = (result, timestamp)
        key = (sql, params)
        ...

facade = CachedDatabaseFacade(DatabaseFacade())
r1 = facade.query('SELECT * FROM users')
r2 = facade.query('SELECT * FROM users')  # z cache
print('Results equal:', r1 == r2)

## 4. 🔹 Wiele fasad

Duze systemy moga miec wiele fasad na roznych poziomach:

- Fasada podsystemu: upraszcza jeden podsystem
- Fasada aplikacji: laczy wiele podsystemow
- Fasada API: punkt wejscia dla zewnetrznych klientow

Hierarchia fasad:
```
ApiFacade
  ├── UserFacade (db, cache, auth)
  ├── OrderFacade (db, payment, email)
  └── ProductFacade (db, cache, search)
```

Fasada kontekstu (contextual facade):
- Tworzona z parametrami kontekstowymi (user_id, tenant_id)
- Kapsulkuje kontekst uzytkownika

> 💡 Wiele fasad nie narusza zasady SRP - kazda odpowiada
> za jeden obszar systemu.

In [ ]:
# Hierarchia fasad dla e-commerce
class ProductRepository:
    def find_all(self) -> list: return [{'id': 1, 'name': 'Widget', 'price': 9.99}]
    def find_by_id(self, pid: int) -> dict: return {'id': pid, 'name': 'Widget', 'price': 9.99}

class CartRepository:
    def __init__(self): self._items = []
    def add(self, product: dict, qty: int) -> None: self._items.append({'product': product, 'qty': qty})
    def get_items(self) -> list: return self._items

class PaymentGateway:
    def charge(self, amount: float) -> bool:
        print(f'Charged: {amount} PLN')
        return True

class EmailSender:
    def send_confirmation(self, items: list, total: float) -> None:
        print(f'Email sent: {len(items)} items, total={total} PLN')

# Fasada sklepu
class ProductFacade:
    def __init__(self): self._repo = ProductRepository()
    def list_products(self) -> list: return self._repo.find_all()
    def get_product(self, pid: int) -> dict: return self._repo.find_by_id(pid)

class CheckoutFacade:
    def __init__(self):
        self._cart = CartRepository()
        self._payment = PaymentGateway()
        self._email = EmailSender()

    def add_to_cart(self, product: dict, qty: int = 1) -> None:
        self._cart.add(product, qty)

    def checkout(self) -> bool:
        items = self._cart.get_items()
        total = sum(i['product']['price'] * i['qty'] for i in items)
        if self._payment.charge(total):
            self._email.send_confirmation(items, total)
            return True
        return False

# Fasada aplikacji laczy podsystemy
class ShopFacade:
    def __init__(self):
        self.products = ProductFacade()
        self.checkout = CheckoutFacade()

shop = ShopFacade()
all_products = shop.products.list_products()
product = shop.products.get_product(1)
shop.checkout.add_to_cart(product, 2)
shop.checkout.checkout()

---

### 🐍 Cwiczenia - wiele fasad

1. Dodaj do `ShopFacade` klase `InventoryFacade` z metodami
   `check_stock(product_id)` i `reserve(product_id, qty)`.
2. Napisz `HotelFacade` z pod-fasadami `RoomFacade`, `RestaurantFacade`,
   `ConciergeFacade`.
3. *(Trudniejsze)* Napisz fasade z kontekstem uzytkownika:
   `UserContextFacade(user_id)` ktora przy kazdej operacji
   sprawdza uprawnienia.

In [ ]:
# Cwiczenie 1: InventoryFacade
class InventoryService:
    def __init__(self): self._stock = {1: 10, 2: 5}
    def get_stock(self, product_id: int) -> int: return self._stock.get(product_id, 0)
    def reserve(self, product_id: int, qty: int) -> bool:
        if self._stock.get(product_id, 0) >= qty:
            self._stock[product_id] -= qty
            return True
        return False

class InventoryFacade:
    def __init__(self): self._svc = InventoryService()
    def check_stock(self, product_id: int) -> int: ...
    def reserve(self, product_id: int, qty: int) -> bool: ...

class ShopFacadeV2:
    def __init__(self):
        self.products = ProductFacade()
        self.checkout = CheckoutFacade()
        self.inventory = InventoryFacade()

shop2 = ShopFacadeV2()
print(f'Stock for product 1: {shop2.inventory.check_stock(1)}')
print(f'Reserved: {shop2.inventory.reserve(1, 3)}')
print(f'Remaining: {shop2.inventory.check_stock(1)}')

In [ ]:
# Cwiczenie 2: HotelFacade
class RoomFacade:
    def book(self, room_type: str, nights: int) -> str:
        return f'Room {room_type} x {nights} nights booked'
    def check_in(self, booking_id: str) -> None:
        print(f'Checked in: {booking_id}')

class RestaurantFacade:
    def reserve_table(self, guests: int, time: str) -> str:
        return f'Table for {guests} at {time}'

class ConciergeFacade:
    def book_taxi(self, destination: str) -> str:
        return f'Taxi to {destination} booked'

class HotelFacade:
    def __init__(self):
        self.rooms = RoomFacade()
        self.restaurant = RestaurantFacade()
        self.concierge = ConciergeFacade()

hotel = HotelFacade()
print(hotel.rooms.book('double', 3))
print(hotel.restaurant.reserve_table(2, '20:00'))
print(hotel.concierge.book_taxi('Airport'))

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: UserContextFacade
class UserPermissions:
    ROLES = {'admin': {'read', 'write', 'delete'}, 'editor': {'read', 'write'}, 'viewer': {'read'}}
    def can(self, user_id: int, action: str) -> bool:
        role = 'editor' if user_id % 2 == 0 else 'viewer'
        return action in self.ROLES.get(role, set())

class ContentService:
    def get(self, content_id: int) -> dict: return {'id': content_id, 'text': 'content'}
    def update(self, content_id: int, data: dict) -> None: print(f'Updated {content_id}')
    def delete(self, content_id: int) -> None: print(f'Deleted {content_id}')

class UserContextFacade:
    # hint: sprawdz uprawnienia przed kazdym wywolaniem,
    # rzuc PermissionError jesli brak uprawnien
    def __init__(self, user_id: int):
        ...

    def get_content(self, content_id: int) -> dict: ...
    def update_content(self, content_id: int, data: dict) -> None: ...
    def delete_content(self, content_id: int) -> None: ...

editor = UserContextFacade(2)    # even -> editor
print(editor.get_content(1))
editor.update_content(1, {'text': 'new'})
try:
    editor.delete_content(1)     # brak uprawnien
except PermissionError as e:
    print(f'Denied: {e}')

## 5. 🔹 Facade vs Adapter vs Mediator

Trzy wzorce sa czesto mylone poniewaz wszystkie wprowadzaja
posrednika miedzy obiektami:

| Wzorzec | Problem | Relacja |
|---|---|---|
| Facade | Uproszczenie zlozonosci | 1 fasada -> wiele podsystemow |
| Adapter | Niezgodnosc interfejsow | 1 adapter -> 1 adaptee |
| Mediator | Zarzadzanie komunikacja | 1 mediator -> wiele kolegow |

**Facade**:
- Motywacja: ulatwienie zycia klientowi
- Podsystemy nie wiedza o fasadzie
- Klient moze nadal uzywac podsystemow bezposrednio

**Adapter**:
- Motywacja: interoperacyjnosc
- Zmienia interfejs pojedynczego obiektu
- Klient widzi tylko nowy interfejs

**Mediator**:
- Motywacja: zmniejszenie zaleznosci miedzy obiektami
- Obiekty komunikuja sie przez mediatora, nie bezposrednio
- Centralny kontroler komunikacji

> 💡 Pytaj o motywacje: uproszczenie? -> Facade.
> Zmiana interfejsu? -> Adapter. Komunikacja? -> Mediator.

In [ ]:
# Porownanie na przykladzie systemu powiadomien

class NotificationSystem:
    def send_sms(self, to: str, msg: str) -> None: print(f'SMS to {to}: {msg}')
    def send_email(self, to: str, subj: str, body: str) -> None: print(f'Email to {to}: {subj}')
    def send_push(self, device_id: str, title: str, body: str) -> None: print(f'Push {device_id}: {title}')

# FACADE: upraszcza zlozone API
class NotificationFacade:
    def __init__(self): self._ns = NotificationSystem()
    def notify(self, user_contact: dict, msg: str) -> None:
        if 'phone' in user_contact: self._ns.send_sms(user_contact['phone'], msg)
        if 'email' in user_contact: self._ns.send_email(user_contact['email'], 'Notification', msg)
        if 'device' in user_contact: self._ns.send_push(user_contact['device'], 'Notification', msg)

# ADAPTER: zmienia stary interfejs na nowy
class OldAlertSystem:
    def trigger_alert(self, level: str, message: str) -> None:
        print(f'[{level}] ALERT: {message}')

class AlertToNotificationAdapter:
    def __init__(self, old: OldAlertSystem):
        self._old = old
    def notify(self, user_contact: dict, msg: str) -> None:
        self._old.trigger_alert('INFO', msg)  # zmiana interfejsu

# MEDIATOR: centralny koordynator
class EventMediator:
    def __init__(self):
        self._handlers: dict = {}
    def subscribe(self, event: str, handler) -> None:
        self._handlers.setdefault(event, []).append(handler)
    def publish(self, event: str, data: dict) -> None:
        for handler in self._handlers.get(event, []):
            handler(data)

contact = {'email': 'alice@x.com', 'phone': '+48123456789'}
print('--- Facade ---')
NotificationFacade().notify(contact, 'Hello!')

print('--- Adapter ---')
AlertToNotificationAdapter(OldAlertSystem()).notify(contact, 'Hello!')

print('--- Mediator ---')
med = EventMediator()
med.subscribe('user.login', lambda d: print(f'Login: {d["user"]}'))
med.subscribe('user.login', lambda d: print(f'Audit: {d}'))
med.publish('user.login', {'user': 'alice'})

---

### 🐍 Cwiczenia - roznice miedzy wzorcami

1. Dla tego samego problemu (`PaymentSystem`) napisz Facade, Adapter
   i Mediator. Opisz co kazdy z nich robi inaczej.
2. Zidentyfikuj w Django (lub Flask) przyklady Facade, Adapter, Mediator.
3. *(Trudniejsze)* Napisz system gdzie jeden obiekt jest jednoczesnie
   Fasada (dla klienta zewnetrznego) i Mediatorem (dla podsystemow).

In [ ]:
# Cwiczenie 1: PaymentSystem - trzy wzorce
class PaymentGatewayRaw:
    def authorize_card(self, card: str, amount: float) -> str: return 'auth_token'
    def capture(self, token: str) -> bool: return True
    def void(self, token: str) -> bool: return True

# Facade: prosty interfejs
class PaymentFacade:
    def __init__(self): self._gw = PaymentGatewayRaw()
    def pay(self, card: str, amount: float) -> bool:
        token = self._gw.authorize_card(card, amount)
        return self._gw.capture(token)

# Adapter: stary interfejs charge()
class LegacyPaymentAdapter:
    def __init__(self, gw: PaymentGatewayRaw): self._gw = gw
    def charge(self, card: str, amount: float) -> bool:  # stary interfejs
        token = self._gw.authorize_card(card, amount)
        return self._gw.capture(token)

# Mediator: koordynuje platnosc + email + log
class PaymentMediator:
    def __init__(self):
        self._gw = PaymentGatewayRaw()
        self._handlers = []
    def on_payment(self, handler) -> None: self._handlers.append(handler)
    def process(self, card: str, amount: float) -> bool:
        token = self._gw.authorize_card(card, amount)
        result = self._gw.capture(token)
        for h in self._handlers:
            h({'amount': amount, 'success': result})
        return result

print('Facade:', PaymentFacade().pay('4111...', 99.99))
print('Adapter:', LegacyPaymentAdapter(PaymentGatewayRaw()).charge('4111...', 99.99))
med = PaymentMediator()
med.on_payment(lambda e: print(f'Email: payment {"ok" if e["success"] else "failed"}'))
med.on_payment(lambda e: print(f'Log: {e}'))
med.process('4111...', 99.99)

In [ ]:
# Cwiczenie 2: Django/Flask przyklady
examples = {
    'Facade': [
        'Django ORM (Model.objects.filter()) - ukrywa SQL, connection pool',
        'Flask app.run() - ukrywa Werkzeug server, socket binding',
    ],
    'Adapter': [
        'Django backends (AUTH_BACKENDS) - adaptuja rozne systemy auth',
        'Django storage backends - adaptuja S3/local do jednego interfejsu',
    ],
    'Mediator': [
        'Django signals - Signal.send() koordynuje niezaleznych sluchaczy',
        'Flask Blinker - sygnaly miedzy komponentami aplikacji',
    ],
}
for pattern, exs in examples.items():
    print(f'\n{pattern}:')
    for e in exs: print(f'  - {e}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: Facade + Mediator w jednym
class ApplicationCore:
    # Dla klienta zewnetrznego: Fasada (prosty interfejs)
    # Dla podsystemow: Mediator (koordynuje komunikacje)

    def __init__(self):
        self._subsystems = {}
        self._events = {}

    def register(self, name: str, subsystem) -> None:
        # hint: rejestruj podsystemy i pozwol im subskrybowac zdarzenia
        self._subsystems[name] = subsystem

    def emit(self, event: str, data: dict) -> None:
        for handler in self._events.get(event, []):
            handler(data)

    def on(self, event: str, handler) -> None:
        self._events.setdefault(event, []).append(handler)

    # Fasada dla klienta zewnetrznego
    def process_order(self, order: dict) -> bool:
        # hint: uzyj emit() do koordynacji wewnetrznej
        ...

core = ApplicationCore()
core.on('order.created', lambda d: print(f'[Email] Order {d["id"]} confirmed'))
core.on('order.created', lambda d: print(f'[Inventory] Reserve {d["items"]}'))
result = core.process_order({'id': 42, 'items': ['Widget x2']})
print(f'Processed: {result}')